In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('delivery_processed1.csv')
df.tail()

In [ ]:
df['weight_kg'].plot(kind='kde')

In [ ]:
df['weight_bucket'] = pd.cut(df['weight_kg'], 
                              bins = [0, 5, 10, 20, 30, 70, float('inf')],
                              labels = ['Light Parcel', 'Standard Parcel', 'Medium Parcel', 'Heavy Parcel', 'Bulk Parcel', 'Jumbo Bulk Parcel'])

X = df[['pickup_prefix', 'delivery_prefix', 
        'product_category', 'weather', 'weight_bucket']]

In [ ]:
y = df['delivery_days']

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
#train test split 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=32)

In [ ]:
X_train.head()

In [ ]:
y_train.tail()

In [ ]:
X_train['weather'].unique()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
ct = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', sparse_output=False), ['weather', 'weight_bucket']),
   ('le',  OneHotEncoder(drop='if_binary', sparse_output=False), ['product_category'])
    
], remainder='passthrough')

In [ ]:
X_train_enc = ct.fit_transform(X_train)
X_test_enc  = ct.transform(X_test)

In [ ]:
X_train_enc.shape

In [ ]:
X_train_enc

In [ ]:
feature_names = ct.get_feature_names_out()
X_train_enc = pd.DataFrame(X_train_enc, columns=feature_names)
X_test_enc  = pd.DataFrame(X_test_enc,  columns=feature_names)

In [ ]:
import xgboost as xgb

In [ ]:
model = xgb.XGBClassifier(objective='multi:softprob', 
                          n_estimators=500,
                          max_depth=8,
                          learning_rate=0.09,
                          subsample=0.8,
                          colsample_bytree=0.8,
                          min_child_weight=3,
                          random_state=42)

In [ ]:
y_train_enc = y_train - 1
y_test_enc  = y_test - 1

model.fit(X_train_enc, y_train_enc)

In [ ]:
X_test = ct.transform(X_test)


In [ ]:
y_pred = model.predict(X_test_enc)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
accuracy_score(y_test_enc, y_pred)

In [ ]:
print(classification_report(y_test_enc, y_pred))

In [ ]:
sample = X_test[0].reshape(1, -1)

proba = model.predict_proba(sample)[0]
actual_days = model.classes_ + 1

for day, prob in zip(actual_days, proba):
    print(f"Day {day}: {round(prob*100, 1)}%")

In [ ]:
new_data = pd.DataFrame([{
    'pickup_prefix':    20,
    'delivery_prefix':  81,
    'product_category': 'DELICATE',
    'weather':          'SUNNY',
    'weight_bucket':    'Medium Parcel'
}])

new_encoded = ct.transform(new_data)
proba = model.predict_proba(new_encoded)[0]

for day, prob in zip(model.classes_ + 1, proba):
    print(f"Day {day}: {round(prob*100, 1)}%")

In [ ]:
model.predict(new_encoded)

In [ ]:
import matplotlib.pyplot as plt
xgb.plot_importance(model, max_num_features=10)
plt.show()

In [ ]:
# Exporting the model and Encoder's pickle value 

import joblib

joblib.dump(model, 'xgb_model.pkl')
joblib.dump(ct, 'column_transformer.pkl')
